In [14]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path: str, json_path: str) -> int:
    count = 0
    results = []

    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            
            for row in reader:
                name = row["이름"]
                student_id = row["학번"]
                
                mid = row["중간"]
                final = row["기말"]
                hw = row["과제"]

                # 결측값 처리
                if mid == "" or final == "" or hw == "":
                    avg = None
                    grade = None
                    logging.info(f"이름: {name}, 결측값이 존재하여 평균 및 등급을 계산하지 않음.")
                    scores = {
                        "중간": int(mid) if mid else None,
                        "기말": int(final) if final else None,
                        "과제": int(hw) if hw else None
                    }
                else:
                    mid = int(mid)
                    final = int(final)
                    hw = int(hw)
                    scores = {"중간": mid, "기말": final, "과제": hw}
                    
                    # 가중 평균 및 등급 계산
                    avg = (mid * 0.3) + (final * 0.5) + (hw * 0.2)
                    if avg >= 90:
                        grade = "A"
                    elif avg >= 80:
                        grade = "B"
                    elif avg >= 70:
                        grade = "C"
                    else:
                        grade = "F"
                        
                    logging.info(f"이름: {name}, 평균: {avg}, 등급: {grade}")

                results.append({
                    "이름": name,
                    "학번": student_id,
                    "점수": scores,
                    "평균": avg,
                    "등급": grade
                })
                count += 1

        # json 파일 쓰기
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)

        return count

    except FileNotFoundError:
        logging.warning(f"파일이 존재하지 않음: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"파일 인코딩 오류: {csv_path}")
        return 0

make_report("scores.csv", "report.json")


# logging message
# INFO:root:이름: 김언어, 평균: 89.5, 등급: B
# INFO:root:이름: 이국문, 평균: 84.4, 등급: B
# INFO:root:이름: 박영문, 평균: 93.5, 등급: A
# INFO:root:이름: 최역사, 결측값이 존재하여 평균 및 등급을 계산하지 않음.

# 점수 중 하나라도 비어 있으면 평균과 등급을 모두 None으로 처리했다. 
# 한글이 깨지지 않도록 encoding="utf-8"을 사용하고, JSON 출력 시 한글이 unicode escape sequence로 변환되는 것을 막기 위해 ensure_ascii=False를 사용했다.
# 파일 오류는 구체적으로 구분하여 FileNotFoundError, UnicodeDecodeError만 처리했다.



INFO:root:이름: 김언어, 평균: 89.5, 등급: B
INFO:root:이름: 이국문, 평균: 84.4, 등급: B
INFO:root:이름: 박영문, 평균: 93.5, 등급: A
INFO:root:이름: 최역사, 결측값이 존재하여 평균 및 등급을 계산하지 않음.


4

In [ ]:
class InvalidJamoError(ValueError):
    pass

def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"문자열이 아님")
    
    if len(c) != 1:
        raise ValueError(f"길이가 1이 아님")
        
    code = ord(c)
    
    if 0x3131 <= code <= 0x314E:
        return "자음"
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"유효하지 않은 한글 자모: '{c}'")

# 테스트 실행 코드
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(f"'{item}': {result}")
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

# InvalidJamoError를 ValueError의 자식으로 만든 이유는 “값은 맞지만 허용되지 않는 경우”를 표현하기에 적절하기 때문이다.
# 자모 판별은 ord()로 유니코드 범위를 확인하여 처리했다.
# 예외는 종류별로 분리해서 처리하여 프로그램이 중단되지 않도록 했다.

#출력 결과
# 'ㄱ': 자음
#'ㅏ': 모음
# 'ㄲ': 자음
# [InvalidJamoError] 유효하지 않은 한글 자모: '가'
# [ValueError] 길이가 1이 아님
# [TypeError] 문자열이 아님
# 'ㅎ': 자음
# 'ㅣ': 모음
# [ValueError] 길이가 1이 아님

'ㄱ': 자음
'ㅏ': 모음
'ㄲ': 자음
[InvalidJamoError] 유효하지 않은 한글 자모: '가'
[ValueError] 길이가 1이 아님
[TypeError] 문자열이 아님
'ㅎ': 자음
'ㅣ': 모음
[ValueError] 길이가 1이 아님
